# Plotly Basics — Complete Guide
Plotly creates **interactive charts** you can zoom, hover, and filter in the browser.
Unlike Matplotlib (static images), every Plotly chart is interactive by default.

In [2]:
# This installs plotly directly into whichever Python kernel this notebook is using
# Running it with ! means it runs as a terminal command inside the notebook
import sys
!{sys.executable} -m pip install plotly pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Import plotly.express — the high-level, beginner-friendly API
# Think of it like seaborn: one-line charts with smart defaults
import plotly.express as px

# Import plotly.graph_objects — the low-level API for full control
# Think of it like matplotlib: more verbose but more customizable
import plotly.graph_objects as go

# make_subplots lets you put multiple charts in one figure
from plotly.subplots import make_subplots

import pandas as pd
import numpy as np

print('All imports successful!')

All imports successful!


---
## 1. Sample Data
We'll use this throughout the notebook so every chart is consistent.

In [4]:
# Seed makes random numbers reproducible — same data every run
np.random.seed(42)

# Build a simple supply-chain style DataFrame
df = pd.DataFrame({
    'month'       : ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    'orders'      : [120, 145, 132, 160, 178, 155, 190, 210, 195, 230, 215, 250],
    'revenue'     : [24000,29000,26400,32000,35600,31000,38000,42000,39000,46000,43000,50000],
    'region'      : ['North','South','East','West','North','South','East','West','North','South','East','West'],
    'product'     : ['Laptop','Router','Server','Storage','Switch','Laptop','Router','Server','Storage','Switch','Laptop','Router'],
    'delivery_days': np.random.normal(loc=7, scale=2, size=12).round(1),
    'status'      : ['Delivered','Delayed','Delivered','Pending','Delivered','Delivered','Delayed','Delivered','Pending','Delivered','Delayed','Delivered']
})

df.head()

,month,orders,revenue,region,product,delivery_days,status
0,Jan,120,24000,North,Laptop,8.0,Delivered
1,Feb,145,29000,South,Router,6.7,Delayed
2,Mar,132,26400,East,Server,8.3,Delivered
3,Apr,160,32000,West,Storage,10.0,Pending
4,May,178,35600,North,Switch,6.5,Delivered


---
## 2. Line Chart — `px.line()`
Best for trends over time. Hover over any point to see exact values.

In [5]:
# px.line() — draws a connected line across x-axis values
# x: the column used for the horizontal axis (usually time)
# y: the column used for the vertical axis (the metric)
# title: chart heading shown at the top
# markers=True: adds a dot at each data point so it's easier to read

fig = px.line(
    df,
    x='month',
    y='orders',
    title='Monthly Orders Trend',
    markers=True
)

# .show() renders the interactive chart in the notebook output
fig.show()

In [6]:
# Multi-line chart — plot two y columns on the same chart
# This is useful when comparing two related metrics side by side

fig = px.line(
    df,
    x='month',
    y=['orders', 'delivery_days'],   # list of columns = multiple lines
    title='Orders vs Delivery Days',
    markers=True
)

fig.show()

---
## 3. Bar Chart — `px.bar()`
Best for comparing categories against each other.

In [7]:
# px.bar() — vertical bar chart
# color: assigns a different color per unique value in that column
# text_auto: shows the bar value as a label on top of each bar

fig = px.bar(
    df,
    x='month',
    y='revenue',
    color='region',          # each region gets its own color
    title='Monthly Revenue by Region',
    text_auto=True           # auto-displays the value on the bar
)

fig.show()

In [8]:
# Grouped bar — barmode='group' places bars side by side instead of stacked
# Use this to compare the same metric across multiple categories

fig = px.bar(
    df,
    x='region',
    y='orders',
    color='product',
    barmode='group',         # 'stack' stacks them, 'group' puts them side by side
    title='Orders by Region and Product (Grouped)'
)

fig.show()

In [9]:
# Horizontal bar chart — just swap x and y, then set orientation='h'
# Horizontal bars are better when category names are long

fig = px.bar(
    df,
    x='revenue',
    y='month',
    orientation='h',         # 'h' = horizontal, 'v' = vertical (default)
    title='Revenue per Month (Horizontal)',
    color='revenue',         # color intensity changes with value
    color_continuous_scale='Blues'
)

fig.show()

---
## 4. Scatter Plot — `px.scatter()`
Best for finding relationships (correlations) between two numeric columns.

In [10]:
# px.scatter() — each row becomes one dot on the chart
# size: makes the dot bigger/smaller based on a column value
# hover_data: extra columns shown when you hover over a dot
# trendline='ols': adds a regression line (needs statsmodels installed)

fig = px.scatter(
    df,
    x='orders',
    y='revenue',
    color='status',          # color each dot by its status
    size='delivery_days',    # bigger dot = more delivery days
    hover_data=['month', 'product'],   # show these on hover
    title='Orders vs Revenue (colored by Status)'
)

fig.show()

---
## 5. Pie Chart — `px.pie()`
Best for showing part-to-whole relationships (shares/percentages).

In [11]:
# Count how many orders exist per status — needed for the pie chart
status_counts = df['status'].value_counts().reset_index()
status_counts.columns = ['status', 'count']

# px.pie() — slices represent each category's share of the total
# values: the numeric column that determines slice size
# names: the categorical column that labels each slice
# hole=0.4: makes it a donut chart — hole=0 is a full pie

fig = px.pie(
    status_counts,
    values='count',
    names='status',
    title='Order Status Distribution',
    hole=0.4,
    color='status',
    color_discrete_map={
        'Delivered': 'green',
        'Delayed':   'red',
        'Pending':   'orange'
    }
)

fig.show()

---
## 6. Histogram — `px.histogram()`
Best for understanding how a numeric column is distributed (spread, shape).

In [12]:
# px.histogram() — divides values into buckets (bins) and counts how many fall in each
# nbins: how many bars to split the data into (more bins = finer detail)
# marginal='rug': adds tick marks above the x-axis showing individual data points

fig = px.histogram(
    df,
    x='delivery_days',
    nbins=10,
    title='Distribution of Delivery Days',
    color='status',          # split histogram by status
    marginal='rug',          # adds a rug plot above the histogram
    opacity=0.75             # slight transparency so overlaps are visible
)

fig.show()

---
## 7. Box Plot — `px.box()`
Shows the median, quartiles, and outliers of a distribution in one chart.

In [13]:
# px.box() — the box shows Q1-Q3 (middle 50% of data)
# The line inside the box is the median
# Dots outside the whiskers are outliers
# points='all': shows every individual data point overlaid on the box

fig = px.box(
    df,
    x='status',
    y='delivery_days',
    color='status',
    points='all',            # 'all', 'outliers', or False
    title='Delivery Days Distribution by Status'
)

fig.show()

---
## 8. Violin Plot — `px.violin()`
Like a box plot but shows the full shape (density) of the distribution.

In [14]:
# px.violin() — the wider the body, the more data points exist there
# box=True: embeds a box plot inside the violin for reference

fig = px.violin(
    df,
    x='region',
    y='orders',
    color='region',
    box=True,                # show box plot inside the violin
    points='all',            # show all raw data points
    title='Order Distribution by Region (Violin)'
)

fig.show()

---
## 9. Heatmap — `go.Heatmap()`
Best for showing a matrix of values — useful for correlations and pivot tables.

In [15]:
# Build a pivot table first — rows = region, columns = product, values = orders
pivot = df.pivot_table(index='region', columns='product', values='orders', aggfunc='sum').fillna(0)

# go.Heatmap is a low-level graph_objects heatmap
# z: the matrix values (2D array)
# x: column labels, y: row labels
# colorscale: which color gradient to use
# text: the values shown inside each cell

fig = go.Figure(
    go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale='YlOrRd',
        text=pivot.values,           # show numbers inside cells
        texttemplate='%{text:.0f}',  # format: no decimal places
        showscale=True
    )
)

fig.update_layout(title='Orders Heatmap: Region vs Product')
fig.show()

---
## 10. Subplots — `make_subplots()`
Combine multiple charts into one figure side by side or stacked.

In [16]:
# make_subplots creates a grid of chart slots
# rows=1, cols=2 means: 1 row, 2 side-by-side columns
# subplot_titles: heading for each slot

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Monthly Orders', 'Monthly Revenue')
)

# Add a bar chart to slot (row=1, col=1)
fig.add_trace(
    go.Bar(x=df['month'], y=df['orders'], name='Orders', marker_color='steelblue'),
    row=1, col=1
)

# Add a line chart to slot (row=1, col=2)
fig.add_trace(
    go.Scatter(x=df['month'], y=df['revenue'], name='Revenue', mode='lines+markers', marker_color='coral'),
    row=1, col=2
)

# update_layout controls the whole figure's title and size
fig.update_layout(
    title_text='Orders and Revenue Side by Side',
    height=400
)

fig.show()

In [17]:
# 2x2 subplot grid — four charts in one figure
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Orders Trend', 'Revenue Trend', 'Delivery Days', 'Status Counts')
)

fig.add_trace(go.Scatter(x=df['month'], y=df['orders'],  mode='lines+markers', name='Orders'),  row=1, col=1)
fig.add_trace(go.Bar(    x=df['month'], y=df['revenue'],                        name='Revenue'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['delivery_days'],                               name='Del Days'),row=2, col=1)

sc = df['status'].value_counts()
fig.add_trace(go.Bar(x=sc.index, y=sc.values, name='Status'), row=2, col=2)

fig.update_layout(title_text='Supply Chain Dashboard (2x2)', height=700, showlegend=False)
fig.show()

---
## 11. 3D Scatter — `px.scatter_3d()`
Adds a third dimension — useful when three numeric variables interact.

In [18]:
# px.scatter_3d() — like scatter but with x, y, AND z axes
# You can rotate the chart by clicking and dragging

fig = px.scatter_3d(
    df,
    x='orders',
    y='revenue',
    z='delivery_days',
    color='status',
    symbol='region',         # different dot shape per region
    title='3D View: Orders, Revenue, Delivery Days'
)

fig.show()

---
## 12. Funnel Chart — `px.funnel()`
Used in supply chain to show how quantities reduce through stages.

In [19]:
# A funnel shows how a total number shrinks at each stage of a process
# Common in: sales pipelines, order fulfilment, warehouse flows

funnel_data = pd.DataFrame({
    'stage': ['Orders Placed', 'Picked from Warehouse', 'Dispatched', 'Out for Delivery', 'Delivered'],
    'count': [250, 230, 215, 200, 190]
})

fig = px.funnel(
    funnel_data,
    x='count',
    y='stage',
    title='Order Fulfilment Funnel'
)

fig.show()

---
## 13. Treemap — `px.treemap()`
Shows hierarchical data as nested rectangles — area = quantity.

In [20]:
# px.treemap() — each rectangle's area is proportional to its value
# path: the hierarchy from outermost to innermost level
# values: what determines the size of each rectangle

fig = px.treemap(
    df,
    path=['region', 'product'],   # region > product hierarchy
    values='orders',
    color='revenue',
    title='Orders Treemap: Region → Product'
)

fig.show()

---
## 14. Customization — Titles, Axes, Colors, Themes
`update_layout()` and `update_traces()` control every visual detail.

In [21]:
fig = px.bar(df, x='month', y='orders', title='Custom Styled Chart')

# update_layout — controls the overall figure appearance
fig.update_layout(
    title=dict(text='Customized Orders Chart', font=dict(size=22, color='navy')),
    xaxis_title='Month',           # x-axis label
    yaxis_title='Total Orders',    # y-axis label
    plot_bgcolor='lightyellow',    # background color inside the chart area
    paper_bgcolor='white',         # background color outside the chart area
    font=dict(family='Arial', size=13),   # global font settings
    showlegend=True,
    height=450
)

# update_traces — controls how the data is drawn (bars, lines, dots)
fig.update_traces(
    marker_color='steelblue',      # all bars become steel blue
    marker_line_color='navy',      # bar border color
    marker_line_width=1.5          # bar border thickness
)

fig.show()

In [22]:
# Plotly has built-in themes (templates) — apply with template=
# Available: 'plotly', 'plotly_white', 'plotly_dark', 'ggplot2',
#            'seaborn', 'simple_white', 'none'

fig = px.line(
    df, x='month', y='revenue',
    title='Dark Theme Revenue Trend',
    markers=True,
    template='plotly_dark'   # dark background theme
)

fig.show()

---
## 15. Adding Annotations and Reference Lines
Use `add_hline()`, `add_vline()`, and `add_annotation()` to mark important values.

In [23]:
fig = px.bar(df, x='month', y='orders', title='Orders with Target Line')

# add_hline: draws a horizontal dashed line at a specific y value
# Useful for showing targets, averages, or thresholds
fig.add_hline(
    y=170,
    line_dash='dash',
    line_color='red',
    annotation_text='Target: 170 orders',
    annotation_position='top right'
)

# add_annotation: places a text label at a specific (x, y) position
fig.add_annotation(
    x='Dec', y=250,
    text='Peak Month!',
    showarrow=True,
    arrowhead=2,
    font=dict(color='green', size=13)
)

fig.show()

---
## 16. Saving Charts
Export as interactive HTML or static PNG/PDF.

In [24]:
fig = px.bar(df, x='month', y='orders', title='Export Example')

# Save as interactive HTML — opens in any browser, fully interactive
# No extra libraries needed for HTML export
fig.write_html('orders_chart.html')
print('Saved: orders_chart.html')

# Save as static PNG — needs: pip install kaleido
# Uncomment the lines below after installing kaleido
# fig.write_image('orders_chart.png')
# fig.write_image('orders_chart.pdf')
# print('Saved PNG and PDF')

Saved: orders_chart.html


---
## Quick Reference Card

| Chart Type | Function | Best Used For |
|---|---|---|
| Line | `px.line()` | Trends over time |
| Bar | `px.bar()` | Category comparisons |
| Scatter | `px.scatter()` | Correlations between two numbers |
| Pie / Donut | `px.pie()` | Part-to-whole shares |
| Histogram | `px.histogram()` | Distribution of one column |
| Box | `px.box()` | Spread + outliers |
| Violin | `px.violin()` | Full distribution shape |
| Heatmap | `go.Heatmap()` | Matrix / pivot table values |
| 3D Scatter | `px.scatter_3d()` | Three numeric dimensions |
| Funnel | `px.funnel()` | Stage-by-stage reduction |
| Treemap | `px.treemap()` | Hierarchical proportions |
| Subplots | `make_subplots()` | Multiple charts in one figure |

### Key Methods
| Method | What it does |
|---|---|
| `fig.show()` | Renders the chart |
| `fig.update_layout()` | Titles, size, background, fonts |
| `fig.update_traces()` | Bar colors, line styles, dot sizes |
| `fig.add_hline()` | Horizontal reference line |
| `fig.add_vline()` | Vertical reference line |
| `fig.add_annotation()` | Text label at a specific point |
| `fig.write_html()` | Save as interactive HTML |
| `fig.write_image()` | Save as PNG/PDF (needs kaleido) |